# sheet_operate — Phase 3 SFT（LoRA）
Qwen3-4B-Instruct-2507 ＋ LoRA r=64 ＋ 通用資料混合（抗遺忘）＋ Drive checkpoint 斷點續訓

**使用步驟**
1. Colab 選 GPU 執行階段（96GB 或 A100/H100 皆可，4B LoRA 需求不高）
2. 填好下方 `CFG`（repo_url 或先把 repo 放到 Drive）
3. 全部執行。session 斷線後重跑全部 cell 會自動從最後一個 checkpoint 續訓
4. 流程：基準線 eval（裸模型）→ SFT → 訓後 eval → 通用能力抽查


In [ ]:
# ===== 設定 =====
CFG = dict(
    repo_url   = "https://github.com/timmytsaa/sheet_operate.git",
    drive_root = "/content/drive/MyDrive/公司/未命名資料夾/sheet_operate",  # checkpoint 與輸出存放處
    run_name   = "sft_v3",        # 本輪訓練代號（checkpoint / adapter / eval csv 都用它命名）
    init_adapter = "ckpt_grpo_v1",   # 熱啟動起點：可給資料夾（自動取最新 checkpoint）或確切路徑；空字串 = 從 base 重練
    base_model = "Qwen/Qwen3-4B-Instruct-2507",
    max_seq_len = 4096,
    lora_r = 64, lora_alpha = 64,
    lr = 2e-4, epochs = 2,
    per_device_bs = 8, grad_accum = 2,      # 有效 batch 16
    mix_general = True, general_n = 110,     # 通用中文指令資料混合量（約 15%）
    run_baseline = True,                     # 訓練前先量裸模型基準線
    eval_limit = None,                       # 想快速煙霧測試可設 24
    seed = 3407,
)

In [ ]:
%%capture
# ===== 安裝依賴 =====
!pip install unsloth
!pip install openpyxl formulas    # formulas：評測公式任務時求值用

In [ ]:
# ===== 掛載 Drive、取得 repo、重生評測任務 =====
import glob, os, shutil, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

os.makedirs(CFG["drive_root"], exist_ok=True)
REPO = "/content/sheet_operate"
MARKER = os.path.join("scripts", "gen_tasks.py")   # repo 完整性檢查點

def find_zip():
    for name in ("repo.zip", "sheet_operate_repo.zip"):
        p = os.path.join(CFG["drive_root"], name)
        if os.path.exists(p):
            return p
    hits = sorted(glob.glob(os.path.join(CFG["drive_root"], "*.zip")))
    return hits[0] if hits else None

def has_marker(path):
    return os.path.exists(os.path.join(path, MARKER))

# repo 已存在也要 pull。舊版是 `and not has_marker(REPO)`，clone 成功後就永遠
# 不再更新——推上去的修正 Colab 這邊拿不到，踩過兩次。
if os.path.exists(REPO) and has_marker(REPO) and CFG["repo_url"]:
    r = subprocess.run(["git", "-C", REPO, "pull"], capture_output=True, text=True)
    out = (r.stdout or r.stderr or "").strip().splitlines()
    print("git pull:", out[-1] if out else "(無輸出)")

if CFG["repo_url"] and not has_marker(REPO):
    try:
        if not os.path.exists(REPO):
            subprocess.run(["git", "clone", CFG["repo_url"], REPO], check=True)
        else:
            subprocess.run(["git", "-C", REPO, "pull"], check=False)
    except Exception as e:
        print("[提示] git clone 失敗，改試 Drive 的 zip：", e)
    if not has_marker(REPO):
        print("[提示] repo_url 抓下來是空倉庫（程式碼從未推上去），改用 Drive 的 zip")
        shutil.rmtree(REPO, ignore_errors=True)

if not has_marker(REPO):
    zp = find_zip()
    if zp:
        import zipfile
        shutil.rmtree(REPO, ignore_errors=True)
        with zipfile.ZipFile(zp) as z:
            z.extractall(REPO)
        print("已解壓：", zp)
    elif has_marker(os.path.join(CFG["drive_root"], "repo")):
        REPO = os.path.join(CFG["drive_root"], "repo")

if not has_marker(REPO):
    print("診斷：repo_url =", repr(CFG["repo_url"]))
    print("診斷：", CFG["drive_root"], "內容 =", os.listdir(CFG["drive_root"]))
    raise RuntimeError(f"找不到可用的 repo：請把 repo.zip 上傳到 Drive 的 {CFG['drive_root']}/，"
                       "或把 CFG['repo_url'] 指向已推上程式碼的倉庫")
sys.path.insert(0, REPO)
os.chdir(REPO)

def run_script(args):
    # 錯誤原文照印，不再吞掉子行程的 stderr
    r = subprocess.run([sys.executable] + args, capture_output=True, text=True)
    if r.stdout:
        print(r.stdout[-600:])
    if r.returncode != 0:
        print(r.stderr[-1500:])
        raise RuntimeError(f"指令失敗（exit {r.returncode}）：{' '.join(args)}")

# 評測任務由 seed 確定性重生（與本機一致）。
# 注意：eval / eval_ood 的家族清單已「凍結」為 v1 十三族（基準線已量，不可再變動）；
#       v2 新家族有獨立的 eval_v2（seed 900003）。
V1_FAMS = ("filter_rows,sort_rows,groupby_summary,compute_column,total_row,format_style,"
           "clean_data,join_lookup,split_concat,top_n,composite,context_rule,large_table")
V2_FAMS = "chain_v2,cross_sheet,format_v2,column_ops,semantic_map,calc_chain"
run_script([os.path.join(REPO, "scripts", "gen_tasks.py"), "--out", "data/tasks/eval",
            "--families", V1_FAMS, "--n", "8", "--seed", "900001"])
run_script([os.path.join(REPO, "scripts", "gen_tasks.py"), "--out", "data/tasks/eval_ood",
            "--families", V1_FAMS, "--n", "5", "--seed", "900002", "--schema", "hr"])
run_script([os.path.join(REPO, "scripts", "gen_tasks.py"), "--out", "data/tasks/eval_v2",
            "--families", V2_FAMS, "--n", "8", "--seed", "900003"])
run_script([os.path.join(REPO, "scripts", "gen_tasks.py"), "--out", "data/tasks/eval_v3",
            "--families", "offset_layout", "--n", "12", "--seed", "900004"])
run_script([os.path.join(REPO, "scripts", "gen_tasks.py"), "--out", "data/tasks/eval_v4",
            "--families", "formula_write", "--n", "12", "--seed", "900005"])
run_script([os.path.join(REPO, "scripts", "gen_tasks.py"), "--out", "data/tasks/eval_v5",
            "--families", "terse_intent", "--n", "14", "--seed", "900006"])
run_script([os.path.join(REPO, "scripts", "gen_tasks.py"), "--out", "data/tasks/eval_v6",
            "--families", "dup_header,misaligned_merge,two_tier_header,pair_group",
            "--n", "4", "--seed", "900016"])
run_script([os.path.join(REPO, "scripts", "gen_tasks.py"), "--out", "data/tasks/eval_v7",
            "--families", "diff_dirty_key,diff_nullkey,diff_dupkey,diff_carry_cols,diff_multicol",
            "--n", "4", "--seed", "900017"])
CKPT_DIR = os.path.join(CFG["drive_root"], "ckpt_" + CFG["run_name"])
print("repo:", REPO)
print("checkpoints:", CKPT_DIR)

In [ ]:
# ===== 載入基礎模型 =====
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = CFG["base_model"],
    max_seq_length = CFG["max_seq_len"],
    dtype          = None,          # 自動（bf16）
    load_in_4bit   = False,         # 96GB 跑 bf16 綽綽有餘
)

In [ ]:
# ===== Gym 評測函式（基準線與訓後共用；逐題：生成→沙盒執行→逐格驗證） =====
import json as _json
from pathlib import Path
from sheetops.encoder import encode_workbook
from sheetops.env import solve_once
from sheetops.executor import extract_code
from sheetops.prompts import SYSTEM_PROMPT, build_user_prompt

# 模型的 generation_config 內建 max_length=262144，與 max_new_tokens 併用會每次生成
# 都噴一行警告，把評測輸出洗掉。設成 None 讓 max_new_tokens 單獨生效。
try:
    model.generation_config.max_length = None
except Exception:
    pass


def run_gym_eval(model, tokenizer, tasks_dir="data/tasks/eval", limit=None, tag=""):
    FastLanguageModel.for_inference(model)
    task_dirs = sorted(p.parent for p in Path(tasks_dir).glob("*/task.json"))
    if limit:
        task_dirs = task_dirs[:limit]
    rows = []
    for i, td in enumerate(task_dirs):
        spec = _json.loads((td / "task.json").read_text(encoding="utf-8"))
        user = build_user_prompt(spec["instruction"],
                                 encode_workbook(td / "start.xlsx"),
                                 spec.get("context", ""))
        prompt = tokenizer.apply_chat_template(
            [{"role": "system", "content": SYSTEM_PROMPT},
             {"role": "user", "content": user}],
            tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=1024, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
        reply = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                                 skip_special_tokens=True)
        code_ = extract_code(reply)
        if code_:
            rep = solve_once(td, code_)
            ok, score = bool(rep["full_match"]), rep["score"]
        else:
            ok, score = False, 0.0
        rows.append({"id": spec["id"], "family": spec["family"], "pass": ok, "score": score})
        if (i + 1) % 12 == 0:
            print(f"  [{tag}] {i+1}/{len(task_dirs)}  pass so far: {sum(r['pass'] for r in rows)}")
    import pandas as pd
    df = pd.DataFrame(rows)
    print(f"\n[{tag}] overall pass@1 = {df['pass'].mean():.1%}  (avg score {df['score'].mean():.3f})")
    print(df.groupby("family")["pass"].mean().sort_values().to_string())
    return df

In [ ]:
# ===== （建議開啟）基準線評測（v1 已有紀錄可關閉；v2 首跑建議開） =====
baseline_v6_df = baseline_v7_df = baseline_df = baseline_ood_df = baseline_v2_df = baseline_v3_df = baseline_v4_df = baseline_v5_df = None
if CFG["run_baseline"]:
    baseline_df = run_gym_eval(model, tokenizer, limit=CFG["eval_limit"], tag="baseline")
    baseline_df.to_csv(os.path.join(CFG["drive_root"], "eval_baseline.csv"), index=False)
    baseline_ood_df = run_gym_eval(model, tokenizer, "data/tasks/eval_ood",
                                   limit=CFG["eval_limit"], tag="baseline-ood")
    baseline_ood_df.to_csv(os.path.join(CFG["drive_root"], "eval_baseline_ood.csv"), index=False)
    baseline_v2_df = run_gym_eval(model, tokenizer, "data/tasks/eval_v2",
                                  limit=CFG["eval_limit"], tag="baseline-v2")
    baseline_v2_df.to_csv(os.path.join(CFG["drive_root"], "eval_baseline_v2.csv"), index=False)
    baseline_v3_df = run_gym_eval(model, tokenizer, "data/tasks/eval_v3",
                                  limit=CFG["eval_limit"], tag="baseline-v3")
    baseline_v3_df.to_csv(os.path.join(CFG["drive_root"], "eval_baseline_v3.csv"), index=False)
    baseline_v4_df = run_gym_eval(model, tokenizer, "data/tasks/eval_v4",
                                  limit=CFG["eval_limit"], tag="baseline-v4")
    baseline_v4_df.to_csv(os.path.join(CFG["drive_root"], "eval_baseline_v4.csv"), index=False)
    baseline_v5_df = run_gym_eval(model, tokenizer, "data/tasks/eval_v5",
                                  limit=CFG["eval_limit"], tag="baseline-v5")
    baseline_v5_df.to_csv(os.path.join(CFG["drive_root"], "eval_baseline_v5.csv"), index=False)
    baseline_v6_df = run_gym_eval(model, tokenizer, "data/tasks/eval_v6",
                                  limit=CFG["eval_limit"], tag="baseline-v6")
    baseline_v6_df.to_csv(os.path.join(CFG["drive_root"], "eval_baseline_v6.csv"), index=False)
    baseline_v7_df = run_gym_eval(model, tokenizer, "data/tasks/eval_v7",
                                  limit=CFG["eval_limit"], tag="baseline-v7")
    baseline_v7_df.to_csv(os.path.join(CFG["drive_root"], "eval_baseline_v7.csv"), index=False)

In [ ]:
# ===== 掛上 LoRA（可從既有 adapter/checkpoint 熱啟動） =====
model = FastLanguageModel.get_peft_model(
    model,
    r = CFG["lora_r"],
    lora_alpha = CFG["lora_alpha"],
    lora_dropout = 0,
    bias = "none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = CFG["seed"],
)

if CFG.get("init_adapter"):
    import glob as _g
    from safetensors.torch import load_file
    from peft.utils import set_peft_model_state_dict

    def _find_init():
        """找熱啟動權重：init_adapter 可以是確切路徑，也可以是含多個 checkpoint-N 的資料夾
        （自動取步數最大者）；找不到時再掃 drive_root 的兄弟資料夾。"""
        roots = [CFG["drive_root"]] + sorted(
            _g.glob(os.path.join(os.path.dirname(CFG["drive_root"].rstrip("/")), "*")))
        for root in roots:
            base = os.path.join(root, CFG["init_adapter"])
            direct = os.path.join(base, "adapter_model.safetensors")
            if os.path.exists(direct):
                return direct
            cks = _g.glob(os.path.join(base, "checkpoint-*", "adapter_model.safetensors"))
            if cks:
                def step(p):
                    name = os.path.basename(os.path.dirname(p))
                    return int(name.rsplit("-", 1)[1]) if name.rsplit("-", 1)[1].isdigit() else -1
                return max(cks, key=step)
        return None

    init_path = _find_init()
    assert init_path, f"找不到熱啟動 adapter：{CFG['init_adapter']}（找不到就把 CFG['init_adapter'] 設成空字串從頭練）"
    set_peft_model_state_dict(model, load_file(init_path))
    print("已從熱啟動載入：", init_path)

In [ ]:
# ===== 組訓練資料：蒸餾軌跡 ＋ 通用中文指令混合（抗遺忘） =====
import os
import random
from datasets import Dataset

import glob as _glob
from sheetops.prompts import SYSTEM_PROMPT as _SYS   # 以「現在」的提示詞為準

random.seed(CFG["seed"])
samples = []
sft_files = (["data/sft/teacher_sft.jsonl"] + sorted(_glob.glob("data/sft/v2_*.jsonl"))
             + sorted(_glob.glob("data/sft/v3_*.jsonl"))
             + sorted(_glob.glob("data/sft/v4_*.jsonl"))
             + sorted(_glob.glob("data/sft/v5_*.jsonl"))
             # v6/v7 明列檔名，不用 glob：glob "v6_*" 會同時抓到過濾前的 v6_colres.jsonl
             # 與過濾後的 v6_colres_gen.jsonl，造成重複計入，還會把品質關卡剔除的樣本放回來
             + ["data/sft/v6_colres_gen.jsonl", "data/sft/v7_diff_gen.jsonl"])
sft_files = [f for f in sft_files if os.path.exists(f)]
# 各批資料是在不同時期的 SYSTEM_PROMPT 下蒸餾的（v6 才加了規則 7）。
# 訓練與推論必須看到同一份系統提示，否則模型學到的是「規則會變動」——這裡統一改寫。
n_sys_fixed = 0
for fp in sft_files:
    n0 = len(samples)
    with open(fp, encoding="utf-8") as f:
        for line in f:
            rec = _json.loads(line)
            msgs = rec["messages"]
            if msgs and msgs[0]["role"] == "system" and msgs[0]["content"] != _SYS:
                msgs = [{"role": "system", "content": _SYS}] + msgs[1:]
                n_sys_fixed += 1
            samples.append(msgs)
    print(f"  {fp}: {len(samples) - n0} 條")
print(f"蒸餾軌跡合計：{len(samples)} 條（其中 {n_sys_fixed} 條的系統提示已對齊到目前版本）")

if CFG["mix_general"]:
    try:
        from datasets import load_dataset
        gen_ds = load_dataset("yentinglin/TaiwanChat", split="train", streaming=True)
        got = 0
        for ex in gen_ds:
            msgs = ex.get("messages") or []
            msgs = [m for m in msgs if m.get("role") in ("user", "assistant")]
            if len(msgs) >= 2 and msgs[0]["role"] == "user":
                samples.append(msgs[:2])
                got += 1
            if got >= CFG["general_n"]:
                break
        print(f"通用資料混入：{got} 條（TaiwanChat）")
    except Exception as e:
        print(f"[警告] 通用資料載入失敗，僅用任務資料續訓：{e}")

random.shuffle(samples)
texts = [tokenizer.apply_chat_template(m, tokenize=False) for m in samples]
train_ds = Dataset.from_dict({"text": texts})
print(f"訓練樣本總數：{len(train_ds)}")


In [ ]:
# ===== 訓練（只對 assistant 回覆算 loss；自動續訓） =====
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
from transformers.trainer_utils import get_last_checkpoint

os.makedirs(CKPT_DIR, exist_ok=True)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = CFG["max_seq_len"],
        output_dir = CKPT_DIR,
        per_device_train_batch_size = CFG["per_device_bs"],
        gradient_accumulation_steps = CFG["grad_accum"],
        num_train_epochs = CFG["epochs"],
        learning_rate = CFG["lr"],
        lr_scheduler_type = "cosine",
        warmup_ratio = 0.03,
        weight_decay = 0.01,
        logging_steps = 10,
        save_steps = 50,
        save_total_limit = 3,
        bf16 = True,
        seed = CFG["seed"],
        report_to = "none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)
last_ckpt = get_last_checkpoint(CKPT_DIR)
print("resume from:", last_ckpt)
trainer.train(resume_from_checkpoint = last_ckpt)

In [ ]:
# ===== 存 LoRA adapter 到 Drive =====
ADAPTER_DIR = os.path.join(CFG["drive_root"], "adapter_" + CFG["run_name"])
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("adapter saved to:", ADAPTER_DIR)
# Phase 5 部署時再 merge：
# model.save_pretrained_merged(os.path.join(CFG["drive_root"], "merged_" + CFG["run_name"]),
#                              tokenizer, save_method="merged_16bit")

In [ ]:
# ===== 訓後評測（v1 / OOD / v2）＋與基準線比較 =====
tagp = CFG["run_name"]
sft_df = run_gym_eval(model, tokenizer, limit=CFG["eval_limit"], tag=tagp)
sft_df.to_csv(os.path.join(CFG["drive_root"], f"eval_{tagp}.csv"), index=False)
sft_ood_df = run_gym_eval(model, tokenizer, "data/tasks/eval_ood",
                          limit=CFG["eval_limit"], tag=tagp + "-ood")
sft_ood_df.to_csv(os.path.join(CFG["drive_root"], f"eval_{tagp}_ood.csv"), index=False)
sft_v2_df = run_gym_eval(model, tokenizer, "data/tasks/eval_v2",
                         limit=CFG["eval_limit"], tag=tagp + "-v2")
sft_v2_df.to_csv(os.path.join(CFG["drive_root"], f"eval_{tagp}_v2.csv"), index=False)
sft_v3_df = run_gym_eval(model, tokenizer, "data/tasks/eval_v3",
                         limit=CFG["eval_limit"], tag=tagp + "-v3")
sft_v3_df.to_csv(os.path.join(CFG["drive_root"], f"eval_{tagp}_v3.csv"), index=False)
sft_v4_df = run_gym_eval(model, tokenizer, "data/tasks/eval_v4",
                         limit=CFG["eval_limit"], tag=tagp + "-v4")
sft_v4_df.to_csv(os.path.join(CFG["drive_root"], f"eval_{tagp}_v4.csv"), index=False)
sft_v5_df = run_gym_eval(model, tokenizer, "data/tasks/eval_v5",
                         limit=CFG["eval_limit"], tag=tagp + "-v5")
sft_v5_df.to_csv(os.path.join(CFG["drive_root"], f"eval_{tagp}_v5.csv"), index=False)

# v6 欄位定位 / v7 差異比對——這一輪新增的家族，最該看的兩軌
sft_v6_df = run_gym_eval(model, tokenizer, "data/tasks/eval_v6",
                         limit=CFG["eval_limit"], tag=tagp + "-v6")
sft_v6_df.to_csv(os.path.join(CFG["drive_root"], f"eval_{tagp}_v6.csv"), index=False)

sft_v7_df = run_gym_eval(model, tokenizer, "data/tasks/eval_v7",
                         limit=CFG["eval_limit"], tag=tagp + "-v7")
sft_v7_df.to_csv(os.path.join(CFG["drive_root"], f"eval_{tagp}_v7.csv"), index=False)

if baseline_df is not None:
    import pandas as pd
    cmp = pd.DataFrame({
        "baseline": baseline_df.groupby("family")["pass"].mean(),
        "sft":      sft_df.groupby("family")["pass"].mean(),
    })
    cmp["gain"] = cmp["sft"] - cmp["baseline"]
    print(cmp.sort_values("gain").to_string())
    print(f"\noverall (v1 in-dist): baseline {baseline_df['pass'].mean():.1%} -> {sft_df['pass'].mean():.1%}")
    if baseline_ood_df is not None:
        print(f"overall (v1 OOD hr): baseline {baseline_ood_df['pass'].mean():.1%} -> {sft_ood_df['pass'].mean():.1%}")
    if baseline_v2_df is not None:
        print(f"overall (v2 難度階梯): baseline {baseline_v2_df['pass'].mean():.1%} -> {sft_v2_df['pass'].mean():.1%}")
    if baseline_v3_df is not None:
        print(f"overall (v3 版型對地): baseline {baseline_v3_df['pass'].mean():.1%} -> {sft_v3_df['pass'].mean():.1%}")
    if baseline_v4_df is not None:
        print(f"overall (v4 公式撰寫): baseline {baseline_v4_df['pass'].mean():.1%} -> {sft_v4_df['pass'].mean():.1%}")
    if baseline_v5_df is not None:
        print(f"overall (v5 簡略指令): baseline {baseline_v5_df['pass'].mean():.1%} -> {sft_v5_df['pass'].mean():.1%}")
print("\nv3（offset_layout）各變體成績看 csv；判讀：v1/ood/v2 不得比前一輪低 5pt 以上（漂移警戒），"
      "v3 應大幅上升（本輪主目標——真實檔案的版型對地）。")

In [ ]:
# ===== 通用能力抽查（遺忘煙霧偵測器，人工看輸出是否還正常） =====
FastLanguageModel.for_inference(model)
for q in ["用兩三句話介紹台北 101。",
          "解釋什麼是複利，並舉一個簡單的例子。",
          "寫一個 Python 函式判斷質數。",
          "客戶來信抱怨出貨延遲，幫我擬一段得體的道歉回覆。"]:
    p = tokenizer.apply_chat_template([{"role": "user", "content": q}],
                                      tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(p, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=256, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    print("Q:", q)
    print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)[:400])
    print("-" * 60)

## 下一步
- pass@1 若明顯超過基準線且 composite/context_rule 有 20%+ 成功率 → 進 Phase 4 GRPO（同一個 Gym 當 reward）
- 若過擬合跡象（train loss 極低但 eval 不動）→ 換前一個 checkpoint 或減 epoch
- 部署：merge → GGUF → 本地 Ollama（Phase 5）
